In [3]:
import dotenv
from dotenv import load_dotenv
import pandas as pd
import requests
import os
import json
load_dotenv()



True

In [19]:


semantic_scholor_url = os.getenv("SEMANTIC_URL")
params = {
    "query": "global workspace theory consciousness",
    "fields": "title,abstract,year,authors,citationCount,openAccessPdf,externalIds,fieldsOfStudy",
    "fieldsOfStudy": "Neuroscience,Psychology,Biology,Computer Science,Philosophy",
    "year": "2010-2026",
    "openAccessPdf": ""   
}

response = requests.get(semantic_scholor_url, params=params)


In [20]:
data = response.json()
df = pd.DataFrame(data)




In [21]:

df = pd.json_normalize(df["data"])


In [22]:
print(df.shape)
print(df.columns)

(118, 18)
Index(['paperId', 'title', 'year', 'citationCount', 'fieldsOfStudy', 'authors',
       'abstract', 'externalIds.DBLP', 'externalIds.PubMedCentral',
       'externalIds.ArXiv', 'externalIds.DOI', 'externalIds.CorpusId',
       'externalIds.PubMed', 'openAccessPdf.url', 'openAccessPdf.status',
       'openAccessPdf.license', 'openAccessPdf.disclaimer', 'externalIds.MAG'],
      dtype='object')


In [24]:
req_cols = ['paperId', 'title', 'year', 'authors', 'externalIds.DOI' , 'openAccessPdf.url','citationCount']
meta_data = df[req_cols]
meta_data
meta_data.to_csv("../../data/metadata.csv", mode='a' , header=True , index=False)



In [25]:
pdf_url = df['openAccessPdf.url']
print(pdf_url[0])
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
Path = "../../data/raw_pdfs"


https://public-pages-files-2025.frontiersin.org/journals/robotics-and-ai/articles/10.3389/frobt.2025.1607190/pdf


In [ ]:
# download pdf
for index , url in pdf_url.dropna().items():
    try:
        res = requests.get(url, headers=headers, stream=True, timeout=15)
        if res.status_code == 200:
            file_path = os.path.join(Path, f"{index}data.pdf")
            with open(file_path, "wb") as pdf_file:
                for chunk in res.iter_content(chunk_size=4096):
                    pdf_file.write(chunk)
            print(f"Success: Index {index} saved ")
        
    except Exception as e:
        print(f"Error downloading index {index} ({url}): {e}")
        